**Import libraries**

In [1]:
!pip install pyarrow==2 awswrangler
!pip install --upgrade google-api-python-client oauth2client
!pip install gspread

In [2]:
from __future__ import print_function

import argparse
import os, sys
import pandas as pd

sys.path.append("/home/ec2-user/SageMaker/ml-demand-forecasting")


# standard libraries
import time
import numpy as np
import random
from datetime import datetime, timedelta, date
from dateutil.relativedelta import relativedelta
from src.io.load_data_c_online import load_data
from src.feature.add_covid_online import add_covid

# from models.add_vaccine_online import add_vacine
from config.project_config import S3_BUCKET
from src.utils.pomelo_utils import Hal
from src.io.loaders import upload_file_to_s3

# main data
from src.queries.query_main_data_th import query_main_data_th
from src.queries.query_main_data_my import query_main_data_my
from src.queries.query_main_data_id import query_main_data_id


# info data
from src.queries.query_info_data_th import query_info_data_th
from src.queries.query_info_data_my import query_info_data_my
from src.queries.query_info_data_id import query_info_data_id

from src.preprocessing.online_processing import transform_data
from src.preprocessing.data_cleaning import clean_data
from src.preprocessing.data_cleaning import fill_new_categories


import pickle
import s3fs
import boto3

import warnings
from sklearn.exceptions import DataConversionWarning
from pandas.core.common import SettingWithCopyWarning

warnings.simplefilter(action="ignore", category=SettingWithCopyWarning)
warnings.filterwarnings(action="ignore", category=DataConversionWarning)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 10000)
pd.set_option("max_colwidth", 10000)

# set up sagemaker environment
# for running processing job
import sagemaker
from sagemaker import get_execution_role

sagemaker_session = sagemaker.Session()

role = get_execution_role()

temp_data_save_path = '../../temp/data_prep/14072022/'
if not os.path.exists(temp_data_save_path):
    os.makedirs(temp_data_save_path)


# Change this!
S3_FOLDER_PATH = 'data_science/dfm/online_clothing_v2/14072022'

/home/ec2-user/anaconda3/envs/python3/lib/python3.6/site-packages/psycopg2/__init__.py:144: UserWarning: The psycopg2 wheel package will be renamed from release 2.8; in order to keep installing from binary please use "pip install psycopg2-binary" instead. For details see: <http://initd.org/psycopg/docs/install.html#binary-install-from-pypi>.
  """)
/home/ec2-user/anaconda3/envs/python3/lib/python3.6/site-packages/boto3/compat.py:88: PythonDeprecationWarning: Boto3 will no longer support Python 3.6 starting May 30, 2022. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.7 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


# 1. Query Data

In [ ]:
print('running Thailand')
main_th = query_main_data_th()
main_th.to_parquet(temp_data_save_path + 'main_th.parquet')

print('running Malaysia')
main_my = query_main_data_my()
main_my.to_parquet(temp_data_save_path + 'main_my.parquet')

print('running Indonesia')
main_id = query_main_data_id()
main_id.to_parquet(temp_data_save_path + 'main_id.parquet')

In [ ]:
main_th = pd.read_parquet(temp_data_save_path + 'main_th.parquet')
main_id = pd.read_parquet(temp_data_save_path + 'main_id.parquet')
main_my = pd.read_parquet(temp_data_save_path + 'main_my.parquet')

In [ ]:
%%time

# unpivot 'net_units_sold','gross_units_sold','gross_revenue_usd','revenue_usd','is_mega_campaign_order'
unpivot_raw_th = (
    pd.wide_to_long(
        main_th.reset_index(),
        stubnames=[
            "net_units_sold",
            "gross_units_sold",
            "gross_revenue_usd",
            "revenue_usd",
            "item_discount_usd",
            "voucher_discount_usd",
            "is_mega_campaign_order",
        ],
        i="index",
        j="week_id",
        sep="_",
        suffix="\w+",
    )
    .reset_index(level=1)
    .reset_index(drop=True)
)

unpivot_raw_my = (
    pd.wide_to_long(
        main_my.reset_index(),
        stubnames=[
            "net_units_sold",
            "gross_units_sold",
            "gross_revenue_usd",
            "revenue_usd",
            "item_discount_usd",
            "voucher_discount_usd",
            "is_mega_campaign_order",
        ],
        i="index",
        j="week_id",
        sep="_",
        suffix="\w+",
    )
    .reset_index(level=1)
    .reset_index(drop=True)
)

unpivot_raw_id = (
    pd.wide_to_long(
        main_id.reset_index(),
        stubnames=[
            "net_units_sold",
            "gross_units_sold",
            "gross_revenue_usd",
            "revenue_usd",
            "item_discount_usd",
            "voucher_discount_usd",
            "is_mega_campaign_order",
        ],
        i="index",
        j="week_id",
        sep="_",
        suffix="\w+",
    )
    .reset_index(level=1)
    .reset_index(drop=True)
)

In [ ]:
del main_th, main_my, main_id 

In [ ]:
unpivot_main_data = pd.concat([unpivot_raw_th, unpivot_raw_my, unpivot_raw_id])
unpivot_main_data.to_parquet(temp_data_save_path + 'unpivot_main_data.parquet')

In [ ]:
del unpivot_raw_th, unpivot_raw_my, unpivot_raw_id

## Get info data

In [ ]:
%%time

# 19min 41s
print('query Thailand')
info_th = query_info_data_th()
info_th.to_parquet(temp_data_save_path + 'info_th.parquet')

print('query Malaysia')
info_my = query_info_data_my()
info_my.to_parquet(temp_data_save_path + 'info_my.parquet')

print('query Indonesia')
info_id = query_info_data_id()
info_id.to_parquet(temp_data_save_path + 'info_id.parquet')

In [ ]:
info_data = pd.concat([info_th, info_my, info_id])
info_data.to_parquet(temp_data_save_path + 'info_data.parquet')

In [ ]:
info_data

In [ ]:
del info_th, info_my, info_id

## Merge info into main

In [ ]:
raw_merged = unpivot_main_data.merge(
    info_data, on=["id_product", "warehouse", "week_id"], how="left"
)
#raw_merged.to_parquet(temp_data_save_path + 'raw_merged.parquet')

## Calculating discount percent

In [ ]:
raw_merged["discount_utilization"] = np.where(
    raw_merged["gross_revenue_usd"] == 0,
    0,
    1 - (raw_merged["revenue_usd"] / raw_merged["gross_revenue_usd"]),
)

raw_merged["item_discount_percent"] = np.where(
    raw_merged["gross_revenue_usd"] == 0,
    0,
    raw_merged["item_discount_usd"] / raw_merged["gross_revenue_usd"],
)

raw_merged["voucher_discount_percent"] = np.where(
    raw_merged["gross_revenue_usd"] == 0,
    0,
    raw_merged["voucher_discount_usd"] / raw_merged["gross_revenue_usd"],
)

In [ ]:
raw_merged

In [ ]:
raw_merged.to_parquet(temp_data_save_path + 'raw_merged.parquet')

In [3]:
%%time
# 19min 49s - 34min 28s


#raw_w_covid = add_covid(raw_merged)

[marketing_data, lookbook_collection_spend] = load_data(
    "s3://hal-bi-bucket/data_science/dfm/excel_files/marketing_spend_data.csv",
    "s3://hal-bi-bucket/data_science/dfm/excel_files/lookbook_spend_collection.csv",
)

# raw_w_covid.to_csv('s3://hal-bi-bucket/data_science/dfm/online_clothing_v2/data/raw.csv', index=False)
marketing_data.to_csv(
    "s3://hal-bi-bucket/data_science/dfm/online_clothing_v2/data/marketing_data.csv",
    index=False,
)
lookbook_collection_spend.to_csv(
    "s3://hal-bi-bucket/data_science/dfm/online_clothing_v2/data/lookbook_collection_spend.csv",
    index=False,
)

train_loobook_pv_dist = pd.read_csv("s3://hal-bi-bucket/data_science/dfm/online_clothing_v2/data/train_loobook_pv_dist.csv")
pv_dist_train = pd.read_csv("s3://hal-bi-bucket/data_science/dfm/online_clothing_v2/data/pv_dist_train.csv")

CPU times: user 124 ms, sys: 25.5 ms, total: 149 ms
Wall time: 709 ms


# 2. Transform and save file

In [4]:
''' Read data in case we ran it before '''

#main_th = pd.read_parquet(temp_data_save_path + 'main_th.parquet')
#main_my = pd.read_parquet(temp_data_save_path + 'main_my.parquet')
#main_id = pd.read_parquet(temp_data_save_path + 'main_id.parquet')

#unpivot_main_data = pd.read_parquet(temp_data_save_path + 'unpivot_main_data.parquet')
#info_data = pd.read_parquet(temp_data_save_path + 'info_data.parquet')
raw_merged = pd.read_parquet(temp_data_save_path + 'raw_merged.parquet')

In [5]:
for cat in ['1', '2', '3']:
    print(f'Chaging category {cat}')
    raw_merged = fill_new_categories(raw_merged, cat)
    print('--------')

Chaging category 1
Unique categories before mapping: <StringArray>
[<NA>, 'Tops', 'Dresses', 'Outerwears', 'Bottoms', 'Swimwear-Bottoms', 'Jumpsuits & Rompers', 'ACTIVEWEAR', 'Swimsuits', 'Swimwear-Tops']
Length: 10, dtype: string
Number of NA rows before mapping missing categories: (1094613, 62)
Unique categories after mapping: <StringArray>
[<NA>, 'Tops', 'Dresses', 'Outerwears', 'Bottoms', 'Swimwear-Bottoms', 'Jumpsuits & Rompers', 'ACTIVEWEAR', 'Swimsuits', 'Swimwear-Tops']
Length: 10, dtype: string
Number of NA rows after mapping missing categories: (564179, 63)
--------
Chaging category 2
Unique categories before mapping: <StringArray>
[<NA>, 'Shirts', 'Midi', 'Jackets', 'Skirts', 'Mini', 'Blouses', 'Knitted Tops', 'Maxi', 'Pants', 'Bustiers', 'Vests', 'Tanks', 'Bikini Bottoms', 'Cardigans', 'Cloaks & Cover Ups', 'Knee Length', 'Jumpsuits', 'Tees', '-', 'Camis', 'Skorts', 'Bodysuits', 'Rompers', 'Shorts', 'Jeans', 'Blazers', 'Sweatshirts /Hoodies', 'Sports Bra', 'Leggings', 'One 

In [6]:
''' Drop NA in Category 1 ==> About 12% data loss '''
raw_merged = raw_merged[~raw_merged.henry_category_1.isna()]

In [7]:
master_join_online = transform_data(raw_merged,lookbook_collection_spend, train_loobook_pv_dist, pv_dist_train)

Calculate country distribution
Calculate style in drop
<StringArray>
['s', 'l', 'normal_drop', 'm']
Length: 4, dtype: string
unique values in small_collection_spend: [  nan  688.  596. 1153. 1142. 1150.  600.]
Add lookbook collection data
Striping columns...
Cleaning strings...
Add mkt pageview dist
Converting columns to strings
Calculating size distribution for each product id
done


In [8]:
master_join_online = clean_data(master_join_online)

In [10]:
master_join_online.to_parquet(temp_data_save_path + 'master_join_online.parquet')

In [11]:
upload_file_to_s3(
    local_path=temp_data_save_path + 'master_join_online.parquet',
    s3_key=f'{S3_FOLDER_PATH}/master_join_online.parquet',
    bucket_name=S3_BUCKET
)

/home/ec2-user/anaconda3/envs/python3/lib/python3.6/site-packages/boto3/compat.py:88: PythonDeprecationWarning: Boto3 will no longer support Python 3.6 starting May 30, 2022. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.7 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


In [ ]:
master_join_online.shape

In [ ]:
master_join_online.isna().sum().head(50)

In [ ]:
del master_join_online

In [ ]:
mj = pd.read_parquet(f's3://{S3_BUCKET}/{S3_FOLDER_PATH}/master_join_online.parquet')
mj.head()

In [ ]:
mj.isna().sum().head(50)